# Advances of the Project

## Title / Objective:
Reproducible Analysis of CMS Open Data: Search for Dark Matter in Association with Top Quarks
(Based on the CMS publication: “Search for dark matter produced in association with a single top quark or a top quark pair in proton–proton collisions at ($\sqrt s = 13 TeV$”).

# Physics Motivation

The Large Hadron Collider (LHC) operates at center-of-mass energies sufficient to probe physics beyond the Standard Model (BSM). A primary goal of the LHC program is to identify the nature of **Dark Matter (DM)**. In many simplified models, DM candidates ($\chi$) interact with Standard Model particles via a new mediator ($\phi$ or $a$). Since these mediators often couple to mass (Yukawa-like coupling), the **top quark**—being the heaviest elementary particle—offers the most sensitive probe for direct DM production.

At the parton level, the interaction produces top quarks accompanied by a mediator, which subsequently decays into a pair of invisible DM particles ($\chi\bar{\chi}$). This results in a final state characterized by heavy flavor quarks and significant **Missing Transverse Momentum ($p_T^{\text{miss}}$)** due to the invisible particles.

The dominant production mechanisms considered are:

* **Gluon Fusion ($t\bar{t}$+DM):**
    $$gg \to t \bar{t}\,\phi \to t \bar{t} + \chi \bar{\chi}$$

* **Single Top Associated Production ($tW$+DM):**
    $$gb \to t W \phi \to t W + \chi \bar{\chi}$$

* **t-channel Production ($tj$+DM):**
    $$qq' \to t j \phi \to t j + \chi \bar{\chi}$$

###  From Partons to Detector Signatures

The experimental signature is determined by the decay of the top quarks ($t \to W b$). The $W$ boson decays either hadronically ($W \to q\bar{q}'$, BR $\approx 67\%$) or leptonically ($W \to \ell \nu$, BR $\approx 33\%$). Consequently, the analysis is categorized into orthogonal **channels** based on the number of isolated charged leptons ($e, \mu$) in the final state:

* **All-Hadronic (AH) Channel (0 leptons):**
    * *Signature:* Multiple jets, b-tags, and large $p_T^{\text{miss}}$.
    * *Pros/Cons:* Highest signal yield, but suffers from overwhelming QCD multijet background and difficult trigger thresholds.

* **Single-Lepton (SL) Channel (1 lepton):**
    * *Signature:* Exactly one isolated lepton, jets, b-tags, and $p_T^{\text{miss}}$.
    * *Pros/Cons:* The "Golden Channel." It offers an optimal balance, suppressing QCD significantly while retaining higher statistics than the dilepton channel.

* **Dilepton (DL) Channel (2 leptons):**
    * *Signature:* Two opposite-sign leptons, jets, and $p_T^{\text{miss}}$.
    * *Pros/Cons:* Cleanest signal region with very low background, but statistically limited by the low branching fraction.

### The Single-Muon Channel

In this workflow, we focus exclusively on the **Single-Muon ($1\mu$) channel**.

This choice is driven by both physics and pedagogical clarity. Experimentally, muons provide a cleaner signature than electrons in the CMS detector, with lower bremsstrahlung rates and robust identification efficiencies. By requiring exactly one high-$p_T$ muon, we can effectively eliminate the QCD background, leaving **$t\bar{t}$+Jets** and **$W$+Jets** as the primary backgrounds to model.

This strategy allows us to demonstrate a complete End-to-End analysis—from object selection to statistical interpretation—in a controlled environment where the interplay between the DM signal (high $p_T^{\text{miss}}$) and Standard Model backgrounds can be clearly studied.

After defining the objective of the project (Reproducible Analysis of CMS Open Data), we discussed in which data format to work — NanoAOD or MiniAOD. We decided to use NanoAOD, because it is lighter and optimized for analysis tasks.

In the most precise version, we work within cernbox/swan, but it can work in any Jupyter environment; the important packages are: uproot and awkward

## Single Lepton

by B.V

We import all the libraries that we are going to use.


In [ ]:
import os
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hist
import requests

import awkward as ak
import uproot
import vector

vector.register_awkward()

# dpoa_io       : plumbing only. No physics.
# dpoa_datasets : physics inputs (samples, cross sections, luminosity).
import dpoa_io
import dpoa_datasets as dd
from dpoa_io import N_FILES

print(f"awkward {ak.__version__} | uproot {uproot.__version__} | vector {vector.__version__}")
print(f"N_FILES = {N_FILES} per dataset")


We will use the **dpoa_workshop_utilities** module to help you access the datasets. The functions it contains are:

The `nanoaod_filenames` is a dictionary with the urls to the file indexes of the root files for every dataset that we will use in the analysis.

The `pretty_print(fields, fmt='40s', require=None, ignore=None)` function allows you to print subsets of keys based on strings that you require or ignore. It will also format that output based on how many characters you want in a column (you are limited to 80 characters per line).

The `build_lumi_mask(lumifile, tree, verbose=False)` function helps you mask (select) the data that's collected from collisions.




In [ ]:
#-------------------------------

import dpoa_workshop
from dpoa_workshop import (
    nanoaod_filenames,
    get_files_for_dataset,
    pretty_print,
    build_lumi_mask
)
#-------------------------------------------

In the drafts related to the papers, the datasets used from 2016 are listed along with their run periods and corresponding luminosities. However, one must be careful with these values, because not all periods are available and the data format differs from the one originally used, as previously noted.”



In [ ]:
# Reference screenshot from the paper (not shipped with the repository).
dpoa_io.show_table("dataset_2016.png", "2016 datasets and run periods")


![png](output_10_0.png)
    



In [ ]:
dpoa_io.show_table("MC_data.png", "Monte Carlo samples")


![png](output_11_0.png)
    

# Building the Ntuple File Index

CMS Open Data provides **file index text files** (`file_index.txt`) for each dataset.  
These contain the actual XRootD paths to the NanoAOD `.root` files, along with metadata such as the number of events per file. Example line:

root://eospublic.cern.ch//eos/opendata/cms/mc/.../nano_1.root nevts=58293

The objective is to collect file paths from multiple URLs, organize them by dataset, and store them in a JSON file, while handling possible download errors so the program keeps running.

### To streamline the workflow:
- We define a function `download_files(url)` that fetches each `file_index.txt` via HTTP and extracts only the ROOT file paths.  
- We loop over all entries in `nanoaod_filenames` (the dictionary we built earlier) and collect the full list of ROOT files per dataset.  
- The result is stored in a new dictionary `ntuples`, which maps *dataset -- list of ROOT file paths*.  
- Finally, we save this as a JSON file (`ntuples.json`) for later reuse


In [ ]:
# dd.build_registry() replaces the broken 'Zvv' entry and adds the samples the
# upstream registry was missing (RUNBOOK PHYS-01, PHYS-06).
ntuples = dd.build_registry(nanoaod_filenames)

with open(dpoa_io.NTUPLES_JSON, "w") as f:
    json.dump(ntuples, f, indent=1)

print()
print(f"{len(ntuples)} datasets indexed.")


    Dataset                        | Part  | Files
    --------------------------------------------------
    met                            | 0     | 7
    met                            | 1     | 4
    met                            | 2     | 21
    SingleMuon                     | 0     | 1
    SingleMuon                     | 1     | 1
    SingleMuon                     | 2     | 40
    SingleMuon                     | 3     | 8
    SingleMuon                     | 4     | 32
    SingleElectron                 | 0     | 6
    SingleElectron                 | 1     | 20
    SingleElectron                 | 2     | 11
    SingleElectron                 | 3     | 43
    ttbar-semileptonic             | 0     | 41
    ttbar-semileptonic             | 1     | 26
    ttbar-semileptonic             | 2     | 23
    ttbar-semileptonic             | 3     | 21
    ttbar-semileptonic             | 4     | 27
    t-channel-top                  | 0     | 4
    t-channel-top                  | 1     | 4
    t-channel-top                  | 2     | 5
    t-channel-top                  | 3     | 11
    t-channel-top                  | 4     | 1
    t-channel-antitop              | 0     | 13
    ttW                            | 0     | 4
    ttW                            | 1     | 4
    ttW                            | 2     | 1
    ttW                            | 3     | 3
    WJets-HT400to600               | 0     | 3
    WJets-HT400to600               | 1     | 1
    WJets-HT400to600               | 2     | 2
    WJets-HT400to600               | 3     | 2
    WJets-HT400to600               | 4     | 3
    WJets-HT100to200               | 0     | 3
    WJets-HT100to200               | 1     | 5
    WJets-HT100to200               | 2     | 3
    WJets-HT100to200               | 3     | 3
    WJets-HT100to200               | 4     | 6
    WJets-HT200to400               | 0     | 9
    WJets-HT200to400               | 1     | 14
    WJets-HT200to400               | 2     | 20
    WJets-HT200to400               | 3     | 8
    WJets-HT200to400               | 4     | 9
    WJets-HT70to100                | 0     | 4
    WJets-HT70to100                | 1     | 3
    WJets-HT70to100                | 2     | 2
    WJets-HT70to100                | 3     | 3
    WJets-HT70to100                | 4     | 7
    DYJets-Zpt200                  | 0     | 10
    WW                             | 0     | 10
    WW                             | 1     | 16
    WW                             | 2     | 12
    WW                             | 3     | 3
    ZZ                             | 0     | 3
    ZZ                             | 1     | 6
    ZZ                             | 2     | 5
    ZZ                             | 3     | 3
    Zvv                            | 0     | 14
    --------------------------------------------------
    ntuples.json creado. Keys: ['met', 'SingleMuon', 'SingleElectron', 'ttbar-semileptonic', 't-channel-top', 't-channel-antitop', 'ttW', 'WJets-HT400to600', 'WJets-HT100to200', 'WJets-HT200to400', 'WJets-HT70to100', 'DYJets-Zpt200', 'WW', 'ZZ', 'Zvv']

Since `ntuples.json` concatenates all files into a single flat list per dataset, the distinction between source URLs (which often represent different periods like **Run2016G** vs **Run2016H**) is lost in the final file.

**Why counting matters:**
By logging the number of files in each part (`len(paths)`), we establish a **positional map**:
1.  It helps us identify which specific period a root file belongs to based on its index in the merged list.
2.  It ensures we can verify that the total file count matches the sum of the individual periods (sanity check).

And we will download important files like the luminosity file.


In [ ]:
# Golden JSON: the list of certified good luminosity sections for 2016.
# (Replaces `!wget`, which is not available on every platform and re-downloaded
# the file on every run.)
lumi_file = dpoa_io.download_lumi_json()


    --2025-11-05 20:43:37--  https://opendata.cern.ch/record/14220/files/Cert_271036-284044_13TeV_Legacy2016_Collisions16_JSON.txt
    Resolving opendata.cern.ch (opendata.cern.ch)... 137.138.6.31, 2001:1458:201:8b::100:1c8
    Connecting to opendata.cern.ch (opendata.cern.ch)|137.138.6.31|:443... connected.
    HTTP request sent, awaiting response... 200 OK
    Length: 11686 (11K) [text/plain]
    Saving to: ‘Cert_271036-284044_13TeV_Legacy2016_Collisions16_JSON.txt’
    
    Cert_271036-284044_ 100%[===================>]  11.41K  --.-KB/s    in 0.001s  
    
    2025-11-05 20:43:38 (10.2 MB/s) - ‘Cert_271036-284044_13TeV_Legacy2016_Collisions16_JSON.txt’ saved [11686/11686]
    


Complete dictionary, in case it's possible to add all the events but it hasn't been found or verified yet...


In [ ]:

XSEC2_PB = {

    
    ##### TOP QUARK #############
    "ttbar-semileptonic": 364.35,  
    "t-channel-top": 136.02,       # ST t-channel top
    "t-channel-antitop": 80.95,    # ST t-channel antitop
    "tW-top": 35.85,               # ST tW top 
    
    #### Rare Top 
    "ttW": 0.2043,  #  to TTWJetsToLNu
    "ttZ": 0.2529,  # to TTZToLLNuNu
    
    ##### WJETS (HT Binned) 
    
    "WJets-HT70to100":    1372.0,
    "WJets-HT100to200":   1345.0, # (Check this value, it seems high to be binned, maybe it's LO)
    "WJets-HT200to400":   359.7,  
    "WJets-HT400to600":   48.91,  
    "WJets-HT600to800":   12.05,
    "WJets-HT800to1200":  5.501,  
    "WJets-HT1200to2500": 1.329,  
    "WJets-HT2500toInf":  0.0322,

    #######DRELL-YAN (DY)
    "DYJets-HT100to200": 147.40,
    "DYJets-HT200to400": 40.99,
    "DYJets-HT400to600": 5.678,
    
    ##### DIBOSON 
    "WW": 118.7, 
    "ZZ": 16.6,   
    "WZ": 47.13  
}

until while these datasets are being used


In [ ]:
# Cross sections [pb], with provenance next to each value in dpoa_datasets.
# Two were wrong (RUNBOOK PHYS-01):
#   Zvv            wrong sample entirely; now ZJetsToNuNu_Zpt-200toInf
#                  _BPSFilter at 0.7333 pb        [XSDB 2026-09-03]
#   DYJets-Zpt200  was 1.27 pb, actually 0.393 pb [XSDB, UL16 row]
# Everything else is note Table 6.
XSEC_PB = dd.XSEC_PB
print(f"{len(XSEC_PB)} cross sections loaded from dpoa_datasets.")


---

This function builds the fileset used by the analysis. It reads the JSON inventory of ntuples, identifies which samples are data or MC, applies an optional file limit for fast debugging, and attaches minimal metadata such as the cross section and number of files. The output is a clean, ready-to-use dictionary that tells the processor exactly which datasets to run over and how they should be treated.



In [ ]:
def construct_fileset(ntuples_json="ntuples.json", limit=None, verbose=True):
    """
    Parses the input JSON inventory and assigns metadata (xsec, is_data).
    
    Args:
        ntuples_json (str): Path to the JSON file containing the file lists.
        limit (int or None): Max number of files to load per process. 
                             Useful for quick debugging (e.g., limit=1).
                             If None, loads all files (production mode).
        verbose (bool): If True, prints a summary table of loaded samples.
    
    Returns:
        dict: A dictionary structured for the processor (Coffea/UpRoot).
    """
    
    with open(ntuples_json) as f:
        info = json.load(f)

    fileset = {}
    
    if verbose:
        print(f"\n{'Name':30} {'Type':>6} {'N Files':>10} {'XSEC [pb]':>12}")
        print("-" * 65)

    for process_name, file_list in info.items():
        
        # Slicing handles None gracefully, but explicit check is clearer for readers
        if limit is not None:
            files_to_use = file_list[:limit]
        else:
            files_to_use = file_list

        #  IDENTIFY TYPE (DATA vs MC) 
        
        if process_name in XSEC_PB:
            is_data = False
            xsec_value = XSEC_PB[process_name]
            proc_type = "MC"
        else:
            is_data = True
            xsec_value = None  # Real data has no theoretical xsec here
            proc_type = "DATA"

        #### BUILD DICTIONARY 
        # Minimal metadata structure to keep it lightweight

        
        fileset[process_name] = {
            "files": files_to_use,
            "metadata": {
                "is_data": is_data,
                "xsec": xsec_value,
                "n_files_loaded": len(files_to_use)
            }
        }

        #  LOGGING ---
        if verbose:
            xsec_str = f"{xsec_value:.2f}" if xsec_value else "-"
            print(f"{process_name:30} {proc_type:>6} {len(files_to_use):>10} {xsec_str:>12}")
    
    return fileset

This line initializes the full list of datasets for the analysis. It loads all ntuples defined in *ntuples.json* (since `limit=None`) and prints a summary of the samples. The resulting `fileset` becomes the central input that tells the processor which data and MC files to process.



In [ ]:
# ntuples_json resolves through dpoa_io.NTUPLES_JSON (repo root), not a bare
# relative path -- a notebook's kernel working directory is wherever the
# .ipynb file sits, which is notebooks/, not the repo root.
fileset = construct_fileset(
    ntuples_json=str(dpoa_io.NTUPLES_JSON),
    limit=None,
    verbose=True
)


    Name                             Type    N Files    XSEC [pb]
    -----------------------------------------------------------------
    met                              DATA         32            -
    SingleMuon                       DATA         82            -
    SingleElectron                   DATA         80            -
    ttbar-semileptonic                 MC        138       364.35
    t-channel-top                      MC         25       136.02
    t-channel-antitop                  MC         13        80.95
    ttW                                MC         12         0.20
    WJets-HT400to600                   MC         11        48.91
    WJets-HT100to200                   MC         20      1345.00
    WJets-HT200to400                   MC         60       359.70
    WJets-HT70to100                    MC         19      1372.00
    DYJets-Zpt200                      MC         10         1.27
    WW                                 MC         41       118.70
    ZZ                                 MC         17        16.60
    Zvv                                MC         14        77.30

We are about to treat the data as a "black box" in the processing loop. But we need to verify the inputs first. We use uproot, which is a Python library that allows us to read CERN ROOT files directly, without needing C++.


In [ ]:
dataset = "SingleMuon" ##just for training

for i, fpath in enumerate(fileset[dataset]["files"][:10]):
    print(f"{i+1:2d}. {fpath}")


     1. root://eospublic.cern.ch//eos/opendata/cms/Run2016H/SingleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/120000/61FC1E38-F75C-6B44-AD19-A9894155874E.root
     2. root://eospublic.cern.ch//eos/opendata/cms/Run2016H/SingleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/1210000/576759DA-4A35-534B-B926-2A9E4A5A7268.root
     3. root://eospublic.cern.ch//eos/opendata/cms/Run2016H/SingleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/130000/0107961B-4308-F845-8F96-E14622BBA484.root
     4. root://eospublic.cern.ch//eos/opendata/cms/Run2016H/SingleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/130000/0DEE1709-0416-F24B-ACB2-C68997CB6465.root
     5. root://eospublic.cern.ch//eos/opendata/cms/Run2016H/SingleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/130000/1C08614E-0C0E-6044-966A-CAF630CAEF8F.root
     6. root://eospublic.cern.ch//eos/opendata/cms/Run2016H/SingleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/130000/1D87B4FB-E31C-9F43-AC21-C32469DE9FC6.root
     7. root://eospublic.cern.ch//eos/opendata/cms/Run2016H/SingleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/130000/1EB443F2-1230-8042-B8AE-FD50329CA59B.root
     8. root://eospublic.cern.ch//eos/opendata/cms/Run2016H/SingleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/130000/2045F967-9F0A-7C46-9946-787B27D56E88.root
     9. root://eospublic.cern.ch//eos/opendata/cms/Run2016H/SingleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/130000/236A04EE-C105-D947-8A2E-F8CC6731644F.root
    10. root://eospublic.cern.ch//eos/opendata/cms/Run2016H/SingleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/130000/370BE877-DA24-DB41-A875-07A86EAB6852.root



In [ ]:
sample = "ttbar-semileptonic"

root_path = fileset[sample]["files"][0]
print("Open:", root_path)

# open_events() retries with exponential backoff: the public eospublic
# redirector intermittently drops connections, and a single transient failure
# should not abort the notebook.
events = dpoa_io.open_events(root_path)

print("# events:", events.num_entries)

all_keys = events.keys()
print(f"Total branches: {len(all_keys)}")


    Open: root://eospublic.cern.ch//eos/opendata/cms/mc/RunIISummer20UL16NanoAODv9/TTToSemiLeptonic_TuneCP5_13TeV-powheg-pythia8/NANOAODSIM/106X_mcRun2_asymptotic_v17-v1/120000/08FCB2ED-176B-064B-85AB-37B898773B98.root
    # events: 1233000
    Total braches : 1504



In [ ]:
if "genWeight" in events.keys():
    gw = events["genWeight"].array(entry_stop=100_000, library="np")
    print(f"genWeight: mean={gw.mean():.3f}, std={gw.std():.3f}, negativos={(gw<0).mean():.2%}")
else:
    print("There is no genWeight")


    genWeight: mean=300.828, std=39.097, negativos=0.42%

Because std > 0 and negativos > 0, you have learned a golden rule for this dataset:

NEVER count events.

Wrong: N = len(events)

Right: N = sum(events.genWeight)

If you just count them (len), you will be wrong by 0.84% (counting the negatives as positives instead of subtracting them).

---
## "Having defined the dataset, we move to data processing. Since the reference analysis has already optimized the kinematic variables, we will adopt their final cuts directly. We will retrieve these specific thresholds from the summary tables and apply the necessary selection techniques to our data."
---


**The workflow proceeds in three main steps:**

1. **Apply the baseline selection**
   We first select events that match the single-lepton topology (electron or muon).
   This removes events that clearly do not belong to the channel and defines the core event structure we will work with.

2. **Filter the events**
   After the baseline, we keep only the events that satisfy these basic criteria.
   This produces a clean and well-defined sample for further analysis.

3. **Apply additional cuts to build CR and SR**
   With the filtered sample, we introduce extra requirements to define:

   * **Control Regions (CR):** used to study and validate the dominant backgrounds.
   * **Signal Region (SR):** where the dark-matter search is performed.

This step-by-step structure mirrors the logic used in the CMS paper and ensures a transparent and reproducible analysis flow.

**So all of this is reflected in the tables, which are directly the cuts we should use, 
and we don't have to explore it ourselves in this case for this guide.**

### Base line

If you read the drafts

#### **Table 10 — Baseline Selection (SL and AH)**

The **minimum or baseline selection** that every event must satisfy **before being assigned to any region** (Signal or Control).

It includes:

* Lepton quality and type (muons, electrons)
* Minimum cuts on **pT** and **η**
* Requirements on **jets** and **b-jets**
* Event-cleaning criteria
* **Additional-lepton veto**

It acts as the **universal first filter** of the analysis.
No event enters the SR or CR without passing this baseline.

---

#### **Tables 11 and 12 — Expected Events After Optimized Cuts (SL and AH)**

Tables showing:

* Expected event yields for each **background**
* Predictions for different **signal models**
* Results after the baseline plus **optimized cuts**

These tables serve **only for internal validation** in the paper:

* Checking consistency between simulated backgrounds and signals
* Showing how many events remain under different signal hypotheses



In [ ]:
dpoa_io.show_table("Table_10.png", "Table 10 - baseline selection")


![png](output_35_0.png)
    

### Single Muon

1.“Load all physics objects (muons, electrons, jets, MET) from the ROOT file and package them as 4-vectors. This prepares the event for the physics selections that follow.”

2.“Apply the baseline selection: require exactly one tight muon, veto additional leptons, and apply the minimal MET requirement. This defines the core single-lepton topology before any SR or CR classification.”

3.“Clean all jets (quality, pT, ΔR from the muon) and classify them as central or forward. Retain only events with at least two central jets, as required by the topology used in the analysis.”

4.Compute the physics variables used throughout the analysis: W transverse mass, jet multiplicities, b-tag multiplicity, minΔφ(jets,MET), forward-jet count, and mT(b). These variables are the basis for the SR/CR definitions and the later shape studies.

5.Package all selected events and computed variables into NumPy arrays. This standardized structure makes it easy to plot distributions, apply further event selection, or feed the results into statistical tools.


In [ ]:
vector.register_awkward()

# --- b tagging (RUNBOOK PHYS-08 / PHYS-10) --------------------------------
# Was 0.2770, the 2017 DeepJet medium WP. This data is UL2016; Run2016H is
# postVFP, whose medium WP is 0.2489.
BTAG_WP_DEEPJET_MEDIUM_UL16POSTVFP = 0.2489   # UL16 postVFP
BTAG_WP_CSVV2_MEDIUM_2016          = 0.8484   # what the paper actually uses

# The paper tags b jets with CSVv2 medium (note Table 10, "CSVM"), so we use
# it here too. NanoAOD exposes it as Jet_btagCSVV2.
#
# Measured effect in the all-hadronic channel (RUNBOOK PHYS-10): every process
# LOSES events (ttbar 0.86x, Z(nunu) 0.81x) -- the two taggers differ by only
# a few percent on light jets, not the factor we first assumed. The switch is
# kept because it is what the paper does, not because it fixes the shape.
#
# Both counts are stored so the choice can be revisited without reprocessing.
BTAG_DISCRIMINANT = "csv"


# --- top pT reweighting (RUNBOOK PHYS-14), note section "Corrections for MC"
# The generated top pT spectrum in ttbar MC is harder than what is observed.
# The note gives the correction explicitly:
#
#     rho(pT) = exp(0.0615 - 0.0005 * pT)          pT at matrix-element level
#     w       = sqrt(rho(pT_1) * rho(pT_2))
#
# It is normalisation-preserving by construction (measured <w> = 1.0005 over
# all events) but NOT shape-preserving: measured <w> = 0.892 for pTmiss > 250,
# because the signal regions select the boosted tail the correction suppresses.
#
# The two generated tops are taken as the last copies (statusFlags bit 13).
# Using the hard-process copies (bit 7) instead gives the same weight to four
# decimal places, and both give exactly two tops in 100 % of events.
TOP_PT_LASTCOPY_BIT = 13


def top_pt_weight(tree, n_events):
    """Per-event sqrt(rho1*rho2); 1.0 for any event without exactly two tops."""
    w = np.ones(n_events)
    if "GenPart_pdgId" not in tree.keys():
        return w
    pdg = tree["GenPart_pdgId"].array()
    sf  = tree["GenPart_statusFlags"].array()
    pt  = tree["GenPart_pt"].array()
    mask = (abs(pdg) == 6) & ((sf & (1 << TOP_PT_LASTCOPY_BIT)) != 0)
    top_pt = pt[mask]
    rho = np.exp(0.0615 - 0.0005 * top_pt)
    ww = ak.to_numpy(ak.fill_none(np.sqrt(ak.prod(rho, axis=1)), 1.0))
    ok = ak.to_numpy(ak.num(top_pt) == 2)
    w[ok] = ww[ok]
    return w

# --- MET noise filters (RUNBOOK PHYS-07), note section 2.1 ----------------
MET_FILTERS = [
    "Flag_goodVertices",
    "Flag_HBHENoiseFilter",
    "Flag_HBHENoiseIsoFilter",
    "Flag_EcalDeadCellTriggerPrimitiveFilter",
    "Flag_globalTightHalo2016Filter",
    "Flag_BadPFMuonFilter",
    "Flag_BadChargedCandidateFilter",
]
MET_FILTERS_DATA_ONLY = ["Flag_eeBadScFilter"]


def _event_quality(tree, met_pt, IS_DATA):
    """MET noise filters, plus the golden-JSON lumi mask on data."""
    filters = list(MET_FILTERS) + (MET_FILTERS_DATA_ONLY if IS_DATA else [])
    ok = ak.ones_like(met_pt, dtype=bool)
    for flag in filters:
        ok = ok & tree[flag].array()
    if IS_DATA:
        ok = ok & build_lumi_mask(str(dpoa_io.LUMI_JSON), tree)
    return ok



def process_file_muon(filename, dataset="Unknown", IS_DATA=False):
    """Single-muon baseline selection for one NanoAOD file.

    PHYSICS IS UNCHANGED from the original notebook. The only difference is
    that this function no longer wraps its body in `try/except: return None`.
    That handler turned any failure — a dropped XRootD connection, a missing
    branch — into `None`, which the caller then passed to `pd.DataFrame(None)`,
    yielding an empty DataFrame. A failed download was therefore
    indistinguishable from a file with no selected events, and the MC
    normalisation was silently wrong. Errors are now caught, counted and
    reported by dpoa_io.process_dataset().
    """
    with dpoa_io.open_events(filename) as tree:
        muons = ak.zip({
            "pt": tree["Muon_pt"].array(),
            "eta": tree["Muon_eta"].array(),
            "phi": tree["Muon_phi"].array(),
            "mass": tree["Muon_mass"].array(),
            "iso": tree["Muon_pfRelIso04_all"].array(),
            "tightId": tree["Muon_tightId"].array(),
            "looseId": tree["Muon_looseId"].array(),
        }, with_name="Momentum4D")

        electrons = ak.zip({
            "pt": tree["Electron_pt"].array(),
            "eta": tree["Electron_eta"].array(),
            "phi": tree["Electron_phi"].array(),
            "mass": tree["Electron_mass"].array(),
            "cutBased": tree["Electron_cutBased"].array(),
        }, with_name="Momentum4D")

        jets = ak.zip({
            "pt": tree["Jet_pt"].array(),
            "eta": tree["Jet_eta"].array(),
            "phi": tree["Jet_phi"].array(),
            "mass": tree["Jet_mass"].array(),
            "jetId": tree["Jet_jetId"].array(),
            "btag": tree["Jet_btagDeepFlavB"].array(),
            "csv":  tree["Jet_btagCSVV2"].array(),
        }, with_name="Momentum4D")

        met_pt = tree["MET_pt"].array()
        met_phi = tree["MET_phi"].array()

        if not IS_DATA:
            gen_weight = tree["genWeight"].array()
            # PHYS-14: only ttbar has generated tops; every other sample gets 1.
            top_w = ak.Array(top_pt_weight(tree, len(met_pt))
                             if dataset.startswith("ttbar") else np.ones(len(met_pt)))
        else:
            gen_weight = ak.ones_like(met_pt)
            top_w = ak.Array(np.ones(len(met_pt)))

        # PHYS-07: MET noise filters + golden JSON
        pass_quality = _event_quality(tree, met_pt, IS_DATA)

    # ----- 2. BASELINE SELECTION (note Table 10) --------------------------
    mu_tight_mask = (muons.pt > 30) & (abs(muons.eta) < 2.4) & (muons.tightId) & (muons.iso < 0.15)
    mu_loose_mask = (muons.pt > 10) & (abs(muons.eta) < 2.4) & (muons.looseId) & (muons.iso < 0.25)
    ele_veto_mask = (electrons.pt > 10) & (abs(electrons.eta) < 2.5) & (electrons.cutBased >= 1)

    good_muons = muons[mu_tight_mask]
    veto_muons = muons[mu_loose_mask]
    veto_electrons = electrons[ele_veto_mask]

    has_1_tight_mu = (ak.num(good_muons) == 1)
    veto_extra_lep = (ak.num(veto_muons) == 1) & (ak.num(veto_electrons) == 0)
    pass_met = met_pt >= 160          # PHYS-03: note Table 10 (was 150)

    event_mask = has_1_tight_mu & veto_extra_lep & pass_met & pass_quality

    good_muons = good_muons[event_mask]
    jets = jets[event_mask]
    met_pt = met_pt[event_mask]
    met_phi = met_phi[event_mask]
    gen_weight = gen_weight[event_mask]
    top_w      = top_w[event_mask]                 # PHYS-14: same mask

    leading_mu = good_muons[:, 0]

    # ----- 3. JET CLEANING ------------------------------------------------
    jet_clean_mask = (
        (jets.pt > 30) &
        (jets.jetId >= 1) &
        (jets.deltaR(leading_mu) > 0.4)
    )
    all_clean_jets = jets[jet_clean_mask]

    central_jets = all_clean_jets[abs(all_clean_jets.eta) < 2.4]
    forward_jets = all_clean_jets[(abs(all_clean_jets.eta) >= 2.4) & (abs(all_clean_jets.eta) < 5.0)]

    has_2_jets = ak.num(central_jets) >= 2

    # ----- 4. FINAL SELECTION ---------------------------------------------
    f_met = met_pt[has_2_jets]
    f_met_phi = met_phi[has_2_jets]
    f_mu = leading_mu[has_2_jets]
    f_central = central_jets[has_2_jets]
    f_forward = forward_jets[has_2_jets]
    f_gen_weight = gen_weight[has_2_jets]
    f_top_w      = ak.to_numpy(top_w[has_2_jets])   # PHYS-14

    # mT_W
    dphi_lep = f_mu.phi - f_met_phi
    mt_w = np.sqrt(2 * f_mu.pt * f_met * (1 - np.cos(dphi_lep)))

    # min DeltaPhi (two leading jets vs MET)
    jets_top2 = f_central[:, :2]
    dphi_jets = abs(jets_top2.phi - f_met_phi)
    dphi_jets = np.mod(dphi_jets + np.pi, 2 * np.pi) - np.pi
    min_dphi = ak.min(abs(dphi_jets), axis=1)

    # PHYS-08 / PHYS-10: the paper's tagger (CSVv2 medium). Both are stored.
    is_btag_deepjet = f_central.btag > BTAG_WP_DEEPJET_MEDIUM_UL16POSTVFP
    is_btag_csv     = f_central.csv  > BTAG_WP_CSVV2_MEDIUM_2016
    is_btag = is_btag_csv if BTAG_DISCRIMINANT == "csv" else is_btag_deepjet
    n_btag = ak.sum(is_btag, axis=1)

    # Forward jet count
    n_fwd = ak.num(f_forward)

    # mT_b (transverse mass of MET and the leading b-jet)
    b_jets = f_central[is_btag]
    has_b = ak.num(b_jets) > 0

    b_jet_padded = ak.fill_none(ak.firsts(b_jets), {"pt": 0, "phi": 0}, axis=0)
    dphi_b = b_jet_padded.phi - f_met_phi
    calc_mt_b = np.sqrt(2 * b_jet_padded.pt * f_met * (1 - np.cos(dphi_b)))

    mt_b_np = ak.to_numpy(calc_mt_b)
    has_b_np = ak.to_numpy(has_b)
    mt_b_np[~has_b_np] = -1.0  # dummy value for events with no b-tag

    # ----- 5. OUTPUT ------------------------------------------------------
    out = {
        "dataset": dataset,
        "lep_pt": ak.to_numpy(f_mu.pt),
        "lep_eta": ak.to_numpy(f_mu.eta),
        "met": ak.to_numpy(f_met),
        "mT_W": ak.to_numpy(mt_w),
        "genWeight": ak.to_numpy(f_gen_weight),
        "topPtWeight": f_top_w,
        "nJet": ak.to_numpy(ak.num(f_central)),
        "nBTag": ak.to_numpy(n_btag),
        "nBTag_deepjet": ak.to_numpy(ak.sum(is_btag_deepjet, axis=1)),
        "nBTag_csv":     ak.to_numpy(ak.sum(is_btag_csv, axis=1)),
        "min_dphi": ak.to_numpy(min_dphi),
        "nForwardJets": ak.to_numpy(n_fwd),
        "mT_b": mt_b_np,
    }

    # PHYS-03: note Table 10 requires nb >= 1 in the baseline. Never applied
    # upstream, so b-free events the paper excludes were kept.
    df = pd.DataFrame(out)
    return df[df["nBTag"] >= 1].reset_index(drop=True)


This section details the three-stage pipeline designed to ingest raw ROOT files and convert them into manageable Parquet datasets for the **Single-Muon Channel**.

#### 🔹 BLOCK 1: Dataset Configuration
Defines the scope of the analysis for the muon channel.

* **Classification:** It strictly separates the input streams into:
    * **Data:** `SingleMuon` (Real experimental data).
    * **Monte Carlo (MC):** All other datasets ($t\bar{t}$, $W$+Jets, etc.).
* **Utility:** This separation allows the downstream logic to automatically determine if **Generator Weights** (required for MC normalization) should be extracted or ignored.

#### 🔹 BLOCK 2: The Orchestrator Function
 Acts as the high-level **Manager Layer**. It abstracts the file I/O and parallelization logic from the physics analysis.
 It does *not* perform physics calculations. Its role is strictly operational: **Loop $\rightarrow$ Collect $\rightarrow$ Assemble $\rightarrow$ Save**.
* **Workflow Steps:**
    1.  **Validation:** Verifies the dataset exists in the global `fileset` registry.
    2.  **Differentiation:** Sets the `is_data` flag based on the dataset name.
    3.  **Iteration:** Loops through the first `n_files` of the requested dataset.
    4.  **Delegation:** Calls the physics kernel (`process_file_muon`) to extract observables from each file.
    5.  **Aggregation:** Concatenates partial results into a unified `pandas.DataFrame`.
    6.  **Persistence:** Serializes the final DataFrame into a binary **Parquet** file in the `output_raw/` directory.

#### 🔹 BLOCK 3: Execution Loop
 Triggers the batch processing.
* **Function:** Iterates through the list defined in Block 1.
* **Control:** It calls the orchestrator for each dataset, enforcing a limit on the number of input files (`N_FILES`) to control the sample size during testing or production runs.


In [ ]:
# MC samples for the muon channel (unchanged physics list).
datasets_muon_mc = [
    'ttbar-semileptonic',   # tt(1l)
    'ttbar-2l2nu',          # tt(2l)  -- PHYS-06, was missing
    'Zvv-Zpt200',           # Z(nunu) -- PHYS-01, upstream had the wrong file
    'WJets-HT70to100', 'WJets-HT100to200', 'WJets-HT200to400',
    'WJets-HT400to600', 'WJets-HT600to800', 'WJets-HT800to1200',
    'WJets-HT1200to2500', 'WJets-HT2500toInf',   # PHYS-06: tail was missing
    't-channel-top', 't-channel-antitop',
    'ttW', 'WW', 'WZ', 'ZZ',        # PHYS-11: WZ was missing entirely
    'Zvv-Zpt100to200',              # PHYS-12: the lower Z pT bin was missing
    'DYJets-Zpt200',
]

# dpoa_io.process_dataset replaces the hand-rolled orchestrator loop. It:
#   * processes files in parallel threads (uproot releases the GIL on network
#     I/O, so this is several times faster than the original serial loop)
#   * retries transient XRootD failures
#   * REPORTS files that failed instead of silently dropping them
#   * caches to output_raw/*.parquet and reuses it on rerun (overwrite=True
#     forces reprocessing)
print(f"=== MUON CHANNEL - MC ({N_FILES} files per dataset) ===")

muon_reports = {}
for ds in datasets_muon_mc:
    _, rep = dpoa_io.process_dataset(
        ds, fileset, process_file_muon, is_data=False, n_files=N_FILES
    )
    muon_reports[ds] = rep


    === STARTING MUON PROCESSING (10 files per dataset) ===
     Processing RAW SingleMuon (Is Data: True)...
     Saved: output_raw/SingleMuon_raw.parquet with 10832 events.
     Processing RAW ttbar-semileptonic (Is Data: False)...
     Saved: output_raw/ttbar-semileptonic_raw.parquet with 108389 events.
     Processing RAW ttW (Is Data: False)...
     Saved: output_raw/ttW_raw.parquet with 85898 events.
     Processing RAW WW (Is Data: False)...
     Saved: output_raw/WW_raw.parquet with 8476 events.
     Processing RAW ZZ (Is Data: False)...
     Saved: output_raw/ZZ_raw.parquet with 100 events.
     Processing RAW Zvv (Is Data: False)...
     Saved: output_raw/Zvv_raw.parquet with 4 events.
     Processing RAW DYJets-Zpt200 (Is Data: False)...
     Saved: output_raw/DYJets-Zpt200_raw.parquet with 2240 events.
     Processing RAW t-channel-top (Is Data: False)...
     Saved: output_raw/t-channel-top_raw.parquet with 6811 events.
     Processing RAW WJets-HT400to600 (Is Data: False)...
     Saved: output_raw/WJets-HT400to600_raw.parquet with 49574 events.

**Note:**

Notice that we could reuse the dictionary built earlier (`fileset`) to automatically loop over all available datasets. However, for debugging and flexibility, a manual list of datasets was defined. At the beginning, only a few datasets were used for quick tests, and later this list was expanded to match the same datasets contained in the `fileset`. This approach allows easier control during development while keeping compatibility with the full processing workflow.




In [ ]:
# Real data: the SingleMuon primary dataset.
print(f"=== MUON CHANNEL - DATA ({N_FILES} files) ===")

_, rep = dpoa_io.process_dataset(
    "SingleMuon", fileset, process_file_muon, is_data=True, n_files=N_FILES
)
muon_reports["SingleMuon"] = rep


    === STARTING MUON PROCESSING (20 files per dataset) ===
     Processing RAW SingleMuon (Is Data: True)...
     Saved: output_raw/SingleMuon_raw.parquet with 25932 events.



# — Why Normalization Is Needed

This section introduces the central problem:
data and MC cannot be compared directly because:

* Data reflects what the detector *actually recorded*.
* MC is generated with arbitrary numbers of events and must be rescaled.

Your introduction sets the conceptual foundation:
**MC must be weighted so that its event yields match the luminosity of the data sample**.

### **What this enables**

The next blocks will:

* Extract necessary metadata
* Compute normalization weights
* Apply them when building histograms

Without this conceptual block, the following code would feel unmotivated.

---



# Extracting Total Generated Weights (`sumGenWeights`)

### **Explanation**

To compute the MC weight

$$
w = \frac{\sigma \cdot L}{N_{\text{gen}}}
$$

we need the denominator:
**the number of generated events**, usually stored as **sum of generated event weights**:

```
Runs/genEventSumw
```

This block reads that information *efficiently* from the `Runs` tree in each file.

### **What this block does for normalization**

* Provides **N_gen ≡ sumGenWeights**
* Without it, normalization would be impossible
* For **data**, you correctly set the weight to **1** (data is never scaled)








In [ ]:
# Sum of generator weights, needed to normalise MC:  w = xsec * L / sumGenWeights
#
# This MUST cover exactly the same files that were processed above. In the
# original notebook the processing used N_FILES and this cell used a separate
# N_FILES_MC literal; when the two drifted apart (20 vs 10) every MC sample was
# normalised by the wrong denominator. Both now read the single N_FILES constant.
DATA_SL = {"SingleMuon", "SingleElectron"}

sum_weights_map = dpoa_io.sum_gen_weights(fileset, DATA_SL, n_files=N_FILES)


    Dataset                        | SumW (Subset)        | Files Read
    ----------------------------------------------------------------------
    ttbar-semileptonic             | 3.72e+09           | 10
    t-channel-top                  | 9.05e+08           | 10
    t-channel-antitop              | 6.51e+08           | 10
    ttW                            | 1.05e+06           | 10
    WJets-HT400to600               | 1.99e+06           | 10
    WJets-HT100to200               | 1.00e+07           | 10
    WJets-HT200to400               | 3.96e+06           | 10
    WJets-HT70to100                | 9.42e+06           | 10
    DYJets-Zpt200                  | 7.51e+03           | 10
    WW                             | 7.56e+06           | 10
    ZZ                             | 8.50e+05           | 10
    Zvv                            | 4.30e+04           | 10


# Defining the Physics Metadata (Cross Sections + Classification)

### **Explanation**

This block assigns:

* **Cross sections (σ)**
* **Dataset category (ttbar, WJets, etc.)**
* **Plotting style (colors)**

This is where you get the numerator of the weight formula:

$$
\sigma \cdot L
$$

### **How it connects to normalization**

You now have:

* **σ** (from `fileset[ds]["metadata"]`)
* **L** (your chosen `LUM`)
* **N_gen** (from previous block)

→ Everything needed to compute **per-event scale factors**.




In [ ]:
# --- Luminosity (RUNBOOK PHYS-02) ----------------------------------------
# Upstream used one LUM = 3400 pb^-1 for every channel, chosen to make plots
# agree rather than derived. Luminosity depends on WHICH data dataset was
# processed and HOW MUCH of it, so one constant cannot serve two channels.
# Open Data exposes only Run2016H = 8.9 fb^-1 (note Table 2).
DATA_SL = {"SingleMuon", "SingleElectron"}

LUMI_BY_CHANNEL = {
    ch: dd.luminosity_pb(ch, fileset[ch]["files"][:N_FILES], verbose=True)
    for ch in ("SingleMuon", "SingleElectron")
}
print(f"\nupstream used 3400.0 pb^-1 for both.")

# Colours sampled from Figure 4 of paper.pdf so the stacks are directly
# comparable; the upstream palette used a different colour for every process.
COLOR_MAP = {
    "ttbar":   "#FFCC33",   # paper: yellow
    "Zvv":     "#6699FF",   # paper: light blue
    "WJets":   "#9933FF",   # paper: purple
    "tX":      "#FF9933",   # paper: orange
    "Zll":     "#009900",   # paper: green
    "Diboson": "#000099",   # paper: dark blue
}


def get_group_info(name):
    """Grouping as in the paper legend: single top and ttV are folded into
    t+X, and Z(nunu) is kept separate from Z(ll)."""
    if "Zvv" in name:                     return "Zvv", r"Z($\nu\nu$) + jets", COLOR_MAP["Zvv"]
    if "DY" in name:                      return "Zll", r"Z($\ell\ell$) + jets", COLOR_MAP["Zll"]
    if "ttbar" in name:                   return "ttbar", r"$t\bar{t}$", COLOR_MAP["ttbar"]
    if "WJets" in name:                   return "WJets", r"W($\ell\nu$) + jets", COLOR_MAP["WJets"]
    if "channel" in name or "tW" in name: return "tX", r"t+X", COLOR_MAP["tX"]
    if "ttW" in name or "ttZ" in name:    return "tX", r"t+X", COLOR_MAP["tX"]
    if name in ["WW", "ZZ", "WZ"]:        return "Diboson", r"VV, VH", COLOR_MAP["Diboson"]
    return "tX", r"t+X", COLOR_MAP["tX"]


def scale_factor(dataset, channel="muon"):
    """MC event weight: xsec * L(channel) / sumGenWeights. None for data."""
    if dataset in DATA_SL:
        return None
    lum = LUMI_BY_CHANNEL["SingleMuon" if channel == "muon" else "SingleElectron"]
    xsec = fileset[dataset]["metadata"]["xsec"]
    sum_w = sum_weights_map.get(dataset, 0.0)
    if not sum_w or xsec is None:
        return None
    return (xsec * lum) / sum_w


print(f"\n{'Dataset':<22} | {'Xsec [pb]':>10} | {'SumGenW':>12} | {'Scale(mu)':>11}")
print("-" * 66)
for ds in fileset:
    if ds in DATA_SL:
        print(f"{ds:<22} | {'-':>10} | {'-':>12} | {'1.00':>11}")
        continue
    xsec = fileset[ds]["metadata"]["xsec"]
    sum_w = sum_weights_map.get(ds, 0.0)
    sc = scale_factor(ds, "muon")
    if sc is not None:
        print(f"{ds:<22} | {xsec:>10.4f} | {sum_w:>12.3e} | {sc:>11.3e}")
    else:
        print(f"{ds:<22} | {str(xsec):>10} | {sum_w:>12.3e} | {'--':>11}")


    Dataset                   | Xsec [pb]  | SumGenWeights   | Scale Factor
    ----------------------------------------------------------------------
    met                       | None       | 1.00e+00        | ERROR (Xsec=None)
    SingleMuon                | -          | -               | 1.00        
    SingleElectron            | -          | -               | 1.00        
    ttbar-semileptonic        | 364.35     | 3.72e+09        | 2.13e-04    
    t-channel-top             | 136.02     | 9.05e+08        | 3.26e-04    
    t-channel-antitop         | 80.95      | 6.51e+08        | 2.70e-04    
    ttW                       | 0.20       | 1.05e+06        | 4.21e-04    
    WJets-HT400to600          | 48.91      | 1.99e+06        | 5.34e-02    
    WJets-HT100to200          | 1345.00    | 1.00e+07        | 2.91e-01    
    WJets-HT200to400          | 359.70     | 3.96e+06        | 1.97e-01    
    WJets-HT70to100           | 1372.00    | 9.42e+06        | 3.16e-01    
    DYJets-Zpt200             | 1.27       | 7.51e+03        | 3.67e-01    
    WW                        | 118.70     | 7.56e+06        | 3.41e-02    
    ZZ                        | 16.60      | 8.50e+05        | 4.24e-02    
    Zvv                       | 77.30      | 4.30e+04        | 3.90e+00    

#### Regarding the luminosity definition, we employ a fractional approach calculated as `LUM = (20/82) * 8900.0`(perdiod H) .This method posits that processing 20 out of 82 files accounts for roughly  of the data. While strictly an estimate—given that individual ROOT files in CMS datasets vary in event content and processing a fixed number of files does not yield a precise linear fraction of the total luminosity—it aligns sufficiently with the observed yields, representing approximately  of the Run 2016 dataset. We intentionally maintain this minimalist strategy to serve as a guide for reproducibility; it allows the analysis to be run efficiently on personal computers for educational purposes, preserving the correct physics structure without the significant overhead of processing the full multi-terabyte dataset.


To produce publication-quality plots—like those found in CMS analyses—we need a function that:

1. **Builds histograms for each MC process**, applying the final per-event normalization weight.
2. **Groups MC samples by physics category** (tt̄, W+jets, Single Top, Diboson, etc.).
3. **Stacks the MC contributions** in the correct order (smallest on top) so that the final shape reflects the Standard Model expectation.
4. **Overlays the real DATA** histogram with error bars.
5. **Applies CMS styling**, including the “Preliminary” tag and integrated luminosity.
6. Ensures the plot is clean, readable, and suitable for analysis notes or a professional report.

This function does exactly that.
By defining it once, we ensure that **every observable (pT, eta, MET, jets, etc.)** is plotted consistently and in the same professional format used in CMS.

---




In [ ]:
import mplhep as hep
import matplotlib.pyplot as plt

plt.style.use(hep.style.CMS)


def plot_grouped_stack(var_name, x_label, x_range, channel_data="SingleMuon",
                       n_bins=30, log_scale=False, dfs=None):
    """Stacked MC vs data histogram.

    `dfs` was previously read from a module-level `all_dfs` global that later
    cells reassigned to the electron dictionary. Re-running an earlier cell
    then silently plotted the wrong channel. It is now an explicit argument;
    it still falls back to `all_dfs` so existing calls keep working.
    """
    if dfs is None:
        dfs = all_dfs

    bins = np.linspace(x_range[0], x_range[1], n_bins + 1)

    grouped_counts = {}
    grouped_info = {}

    for name, df in dfs.items():
        if name in DATA_SL:
            continue

        group_key, label, color = get_group_info(name)
        counts, _ = np.histogram(df[var_name], bins=bins, weights=df["final_weight"])

        if group_key not in grouped_counts:
            grouped_counts[group_key] = counts
            grouped_info[group_key] = {"label": label, "color": color, "yield": np.sum(counts)}
        else:
            grouped_counts[group_key] += counts
            grouped_info[group_key]["yield"] += np.sum(counts)

    # Order groups by yield: smallest on top of the stack
    active_groups = sorted(grouped_info, key=lambda g: grouped_info[g]["yield"])

    mc_counts, mc_colors, mc_labels = [], [], []
    total_mc = np.zeros(n_bins)

    for g in active_groups:
        mc_counts.append(grouped_counts[g])
        mc_colors.append(grouped_info[g]["color"])
        mc_labels.append(grouped_info[g]["label"])
        total_mc += grouped_counts[g]

    df_data = dfs.get(channel_data)
    if df_data is None:
        print(f"ERROR: No DATA found for channel {channel_data}")
        return

    data_counts, _ = np.histogram(df_data[var_name], bins=bins)

    fig, ax = plt.subplots(figsize=(10, 8))

    if mc_counts:
        hep.histplot(mc_counts, bins=bins, stack=True, histtype="fill",
                     color=mc_colors, label=mc_labels,
                     edgecolor="black", linewidth=1, ax=ax)

    hep.histplot(data_counts, bins=bins, histtype="errorbar", color="black",
                 label=f"{channel_data} (Data)", yerr=True,
                 marker="o", markersize=5, ax=ax)

    # Luminosity is per channel (PHYS-02), so the CMS label shows the value
    # for whichever dataset is being plotted rather than one global constant.
    _lumi = LUMI_BY_CHANNEL.get(channel_data, max(LUMI_BY_CHANNEL.values()))
    hep.cms.label("Preliminary", data=True, lumi=round(_lumi / 1000.0, 1), year=2016, ax=ax)

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], fontsize=16, ncol=2, loc="upper right")
    ax.set_xlabel(x_label, fontsize=24)
    ax.set_ylabel("Events", fontsize=24)
    ax.set_xlim(x_range)

    ceiling = max(np.max(data_counts), np.max(total_mc), 1.0)
    if log_scale:
        ax.set_yscale("log")
        ax.set_ylim(0.1, ceiling * 500)
    else:
        ax.set_ylim(0, ceiling * 1.5)

    plt.tight_layout()
    plt.show()


This block loads every muon-channel dataset from the previously generated _raw.parquet files, applies the correct normalization (data vs. Monte Carlo), assigns a final per-event weight, and stores everything inside a unified dictionary (all_dfs).
Once all datasets are normalized, we can safely call plot_grouped_stack to generate stacked MC vs. Data plots that follow a professional CMS-style presentation.


In [ ]:
def load_channel(channel="muon"):
    """Load the processed parquet files for one channel and attach event weights."""
    data_name = "SingleMuon" if channel == "muon" else "SingleElectron"
    suffix = "_raw.parquet" if channel == "muon" else "_electron_raw.parquet"

    datasets = [data_name] + datasets_muon_mc

    loaded = {}
    missing = []

    for ds in datasets:
        path = dpoa_io.OUTPUT_DIR / f"{ds}{suffix}"
        if not path.exists():
            missing.append(ds)
            continue

        df = pd.read_parquet(path)

        if ds in DATA_SL:
            df["final_weight"] = 1.0
        else:
            scale = scale_factor(ds, channel)
            if scale is None:
                missing.append(f"{ds} (no scale factor)")
                continue
            w = df["genWeight"] * scale if "genWeight" in df.columns else scale
            # PHYS-14: top pT reweighting. Older parquets predate the column,
            # so its absence means "no correction", not an error.
            if "topPtWeight" in df.columns:
                w = w * df["topPtWeight"]
            df["final_weight"] = w

        loaded[ds] = df

    print(f"Loaded {len(loaded)} datasets for the {channel} channel.")
    if missing:
        # The original code swallowed FileNotFoundError with a bare `continue`,
        # so a stack could be missing half its backgrounds with no indication.
        print(f"NOT loaded ({len(missing)}): {', '.join(missing)}")

    return loaded


muon_dfs = load_channel("muon")
all_dfs = muon_dfs

if muon_dfs:
    plot_grouped_stack("mT_W",  r"$m_T^W$ [GeV]",        (0, 200),   "SingleMuon", dfs=muon_dfs)
    plot_grouped_stack("met",   r"$p_T^{miss}$ [GeV]",   (150, 500), "SingleMuon", dfs=muon_dfs, log_scale=True)
    plot_grouped_stack("nJet",  r"$N_{jets}$",           (2, 10),    "SingleMuon", dfs=muon_dfs, n_bins=8, log_scale=True)
    plot_grouped_stack("nBTag", r"$N_{b-tags}$",         (0, 5),     "SingleMuon", dfs=muon_dfs, n_bins=5, log_scale=True)
else:
    print("No data loaded. Run the processing cells above first.")


     Loaded: SingleMuon
     Loaded: ttbar-semileptonic
     Loaded: ttW
     Loaded: WW
     Loaded: ZZ
     Loaded: Zvv
     Loaded: DYJets-Zpt200
     Loaded: t-channel-top
     Loaded: t-channel-antitop
     Loaded: WJets-HT400to600
     Loaded: WJets-HT100to200
     Loaded: WJets-HT200to400
     Loaded: WJets-HT70to100



![png](output_56_1.png)
    



![png](output_56_2.png)
    



![png](output_56_3.png)
    



![png](output_56_4.png)
    

Fix : If we look at the Transverse Mass ($m_T$) plot, there is a distinct peak in the Data at 80 GeV. This corresponds exactly to the W boson resonance ($M_W \approx 80.4$ GeV).The Cause: This resonance peak is dominated by "bulk" events with lower hadronic activity (Low $H_T$). However, we are currently only using the WJets-HT400to600 dataset, which represents the high-energy tail.The Consequence: By missing the HT-100to200 and HT-200to400 datasets, we are effectively cutting out the events that populate this resonance. This explains why the Simulation peak is drastically smaller than the Data peak.

## Electron

To adapt the muon channel for the electron channel, we switch the signal object from `Muon` to `Electron` and adjust the kinematic cuts: pseudorapidity is restricted to **$|\eta| < 2.1$** (instead of 2.4), and a **"Gap Veto"** is implemented to exclude electrons in the barrel-endcap transition region ($1.4442 < |\eta| < 1.566$). Furthermore, the identification criteria change from specific `tightId` and isolation cuts to the standard **`cutBased == 4`** (Tight ID), and **Jet Cleaning** is re-evaluated by requiring $\Delta R > 0.4$ with respect to the selected electron rather than the muon.


In [ ]:
def process_electron(filename, dataset="Unknown", IS_DATA=False):
    """Single-electron baseline selection for one NanoAOD file.

    PHYSICS IS UNCHANGED from the original notebook. The only difference is
    that the file is opened through dpoa_io.open_events(), which retries
    transient XRootD failures, instead of a bare uproot.open().
    """
    with dpoa_io.open_events(filename) as tree:
        electrons = ak.zip({
            "pt": tree["Electron_pt"].array(),
            "eta": tree["Electron_eta"].array(),
            "phi": tree["Electron_phi"].array(),
            "mass": tree["Electron_mass"].array(),
            "cutBased": tree["Electron_cutBased"].array(),
        }, with_name="Momentum4D")

        muons = ak.zip({
            "pt": tree["Muon_pt"].array(),
            "eta": tree["Muon_eta"].array(),
            "phi": tree["Muon_phi"].array(),
            "mass": tree["Muon_mass"].array(),
            "looseId": tree["Muon_looseId"].array(),
            "iso": tree["Muon_pfRelIso04_all"].array(),
        }, with_name="Momentum4D")

        jets = ak.zip({
            "pt": tree["Jet_pt"].array(),
            "eta": tree["Jet_eta"].array(),
            "phi": tree["Jet_phi"].array(),
            "mass": tree["Jet_mass"].array(),
            "jetId": tree["Jet_jetId"].array(),
            "btag": tree["Jet_btagDeepFlavB"].array(),
            "csv":  tree["Jet_btagCSVV2"].array(),
        }, with_name="Momentum4D")

        met_pt = tree["MET_pt"].array()
        met_phi = tree["MET_phi"].array()

        if not IS_DATA:
            gen_weight = tree["genWeight"].array()
            # PHYS-14: only ttbar has generated tops; every other sample gets 1.
            top_w = ak.Array(top_pt_weight(tree, len(met_pt))
                             if dataset.startswith("ttbar") else np.ones(len(met_pt)))
        else:
            gen_weight = ak.ones_like(met_pt)
            top_w = ak.Array(np.ones(len(met_pt)))

        # PHYS-07: MET noise filters + golden JSON
        pass_quality = _event_quality(tree, met_pt, IS_DATA)

    # ----- 1. SELECTION (note Table 10) -----------------------------------
    in_gap = (abs(electrons.eta) > 1.4442) & (abs(electrons.eta) < 1.566)
    ele_tight_mask = (electrons.pt > 30) & (abs(electrons.eta) < 2.1) & (electrons.cutBased == 4) & (~in_gap)
    ele_veto_mask = (electrons.pt > 10) & (electrons.cutBased >= 1)
    mu_loose_mask = (muons.pt > 10) & (abs(muons.eta) < 2.4) & (muons.looseId) & (muons.iso < 0.25)

    good_ele = electrons[ele_tight_mask]
    veto_ele = electrons[ele_veto_mask]
    veto_mu = muons[mu_loose_mask]

    has_1_tight = (ak.num(good_ele) == 1)
    is_exclusive = (ak.num(veto_ele) == 1) & (ak.num(veto_mu) == 0)
    pass_met = met_pt >= 160          # PHYS-03: note Table 10 (was 150)

    event_mask = has_1_tight & is_exclusive & pass_met & pass_quality

    good_ele = good_ele[event_mask]
    jets = jets[event_mask]
    met_pt = met_pt[event_mask]
    met_phi = met_phi[event_mask]
    gen_weight = gen_weight[event_mask]
    top_w      = top_w[event_mask]                 # PHYS-14: same mask
    leading_ele = good_ele[:, 0]

    # ----- 2. JET CLEANING against the ELECTRON ---------------------------
    jet_clean_mask = (
        (jets.pt > 30) & (jets.jetId >= 1) &
        (jets.deltaR(leading_ele) > 0.4)
    )
    all_clean_jets = jets[jet_clean_mask]

    central_jets = all_clean_jets[abs(all_clean_jets.eta) < 2.4]
    forward_jets = all_clean_jets[(abs(all_clean_jets.eta) >= 2.4) & (abs(all_clean_jets.eta) < 5.0)]

    has_2_jets = ak.num(central_jets) >= 2

    f_met = met_pt[has_2_jets]
    f_met_phi = met_phi[has_2_jets]
    f_ele = leading_ele[has_2_jets]
    f_central = central_jets[has_2_jets]
    f_forward = forward_jets[has_2_jets]
    f_gen_weight = gen_weight[has_2_jets]
    f_top_w      = ak.to_numpy(top_w[has_2_jets])   # PHYS-14

    # mT
    dphi = f_ele.phi - f_met_phi
    mt = np.sqrt(2 * f_ele.pt * f_met * (1 - np.cos(dphi)))

    # min DeltaPhi
    jets_top2 = f_central[:, :2]
    dphi_jets = abs(jets_top2.phi - f_met_phi)
    dphi_jets = np.mod(dphi_jets + np.pi, 2 * np.pi) - np.pi
    min_dphi = ak.min(abs(dphi_jets), axis=1)

    # B-tagging - PHYS-08: UL16 postVFP WP (was 0.2770, a 2017 value)
    # PHYS-08 / PHYS-10: the paper's tagger (CSVv2 medium). Both are stored.
    is_btag_deepjet = f_central.btag > BTAG_WP_DEEPJET_MEDIUM_UL16POSTVFP
    is_btag_csv     = f_central.csv  > BTAG_WP_CSVV2_MEDIUM_2016
    is_btag = is_btag_csv if BTAG_DISCRIMINANT == "csv" else is_btag_deepjet
    n_btag = ak.sum(is_btag, axis=1)

    n_fwd = ak.num(f_forward)

    b_jets = f_central[is_btag]
    has_b = ak.num(b_jets) > 0
    b_jet_padded = ak.fill_none(ak.firsts(b_jets), {"pt": 0, "phi": 0}, axis=0)
    dphi_b = b_jet_padded.phi - f_met_phi
    calc_mt_b = np.sqrt(2 * b_jet_padded.pt * f_met * (1 - np.cos(dphi_b)))

    mt_b_np = ak.to_numpy(calc_mt_b)
    has_b_np = ak.to_numpy(has_b)
    mt_b_np[~has_b_np] = -1.0

    out = {
        "dataset": dataset,
        "lep_pt": ak.to_numpy(f_ele.pt),
        "lep_eta": ak.to_numpy(f_ele.eta),
        "met": ak.to_numpy(f_met),
        "mT_W": ak.to_numpy(mt),
        "genWeight": ak.to_numpy(f_gen_weight),
        "topPtWeight": f_top_w,
        "nJet": ak.to_numpy(ak.num(f_central)),
        "nBTag": ak.to_numpy(n_btag),
        "nBTag_deepjet": ak.to_numpy(ak.sum(is_btag_deepjet, axis=1)),
        "nBTag_csv":     ak.to_numpy(ak.sum(is_btag_csv, axis=1)),
        "min_dphi": ak.to_numpy(min_dphi),
        "nForwardJets": ak.to_numpy(n_fwd),
        "mT_b": mt_b_np,
    }

    # PHYS-03: note Table 10 requires nb >= 1 in the baseline. Never applied
    # upstream, so b-free events the paper excludes were kept.
    df = pd.DataFrame(out)
    return df[df["nBTag"] >= 1].reset_index(drop=True)


and with the same idea as the muon one


In [ ]:
datasets_electron_mc = [
    'ttbar-semileptonic',   # tt(1l)
    'ttbar-2l2nu',          # tt(2l)  -- PHYS-06, was missing
    'Zvv-Zpt200',           # Z(nunu) -- PHYS-01, upstream had the wrong file
    'WJets-HT70to100', 'WJets-HT100to200', 'WJets-HT200to400',
    'WJets-HT400to600', 'WJets-HT600to800', 'WJets-HT800to1200',
    'WJets-HT1200to2500', 'WJets-HT2500toInf',   # PHYS-06: tail was missing
    't-channel-top', 't-channel-antitop',
    'ttW', 'WW', 'WZ', 'ZZ',        # PHYS-11: WZ was missing entirely
    'Zvv-Zpt100to200',              # PHYS-12: the lower Z pT bin was missing
    'DYJets-Zpt200',
]

print(f"=== ELECTRON CHANNEL - MC ({N_FILES} files per dataset) ===")

electron_reports = {}
for ds in datasets_electron_mc:
    _, rep = dpoa_io.process_dataset(
        ds, fileset, process_electron, is_data=False,
        n_files=N_FILES, suffix="_electron_raw",
    )
    electron_reports[ds] = rep


    === STARTING ELECTRON PROCESSING (10 files per dataset) ===
     Processing RAW Electron SingleElectron (Is Data: True)...
    Saved: output_raw/SingleElectron_electron_raw.parquet with 10757 events.
     Processing RAW Electron ttbar-semileptonic (Is Data: False)...
    Saved: output_raw/ttbar-semileptonic_electron_raw.parquet with 78228 events.
     Processing RAW Electron ttW (Is Data: False)...
    Saved: output_raw/ttW_electron_raw.parquet with 60974 events.
     Processing RAW Electron WW (Is Data: False)...
    Saved: output_raw/WW_electron_raw.parquet with 6141 events.
     Processing RAW Electron ZZ (Is Data: False)...
    Saved: output_raw/ZZ_electron_raw.parquet with 83 events.
     Processing RAW Electron Zvv (Is Data: False)...
    Saved: output_raw/Zvv_electron_raw.parquet with 24 events.
     Processing RAW Electron DYJets-Zpt200 (Is Data: False)...
    Saved: output_raw/DYJets-Zpt200_electron_raw.parquet with 1398 events.
     Processing RAW Electron t-channel-top (Is Data: False)...
    Saved: output_raw/t-channel-top_electron_raw.parquet with 4238 events.
     Processing RAW Electron WJets-HT400to600 (Is Data: False)...
    Saved: output_raw/WJets-HT400to600_electron_raw.parquet with 35219 events.



In [ ]:
print(f"=== ELECTRON CHANNEL - DATA ({N_FILES} files) ===")

_, rep = dpoa_io.process_dataset(
    "SingleElectron", fileset, process_electron, is_data=True,
    n_files=N_FILES, suffix="_electron_raw",
)
electron_reports["SingleElectron"] = rep


    === STARTING ELECTRON PROCESSING (20 files per dataset) ===
     Processing RAW Electron SingleElectron (Is Data: True)...
    Saved: output_raw/SingleElectron_electron_raw.parquet with 20948 events.



In [ ]:
electron_dfs = load_channel("electron")

if electron_dfs:
    plot_grouped_stack("mT_W",  r"$m_T^W$ [GeV]",      (0, 200),   "SingleElectron", dfs=electron_dfs)
    plot_grouped_stack("met",   r"$p_T^{miss}$ [GeV]", (150, 500), "SingleElectron", dfs=electron_dfs, log_scale=True)
    plot_grouped_stack("nJet",  r"$N_{jets}$",         (2, 10),    "SingleElectron", dfs=electron_dfs, n_bins=8, log_scale=True)
    plot_grouped_stack("nBTag", r"$N_{b-tags}$",       (0, 5),     "SingleElectron", dfs=electron_dfs, n_bins=5, log_scale=True)
else:
    print("No electron data loaded. Run the electron processing cells above first.")


     Cargado: SingleElectron
     Cargado: ttbar-semileptonic
     Cargado: ttW
     Cargado: WW
     Cargado: ZZ
     Cargado: Zvv
     Cargado: DYJets-Zpt200
     Cargado: t-channel-top
     Cargado: t-channel-antitop
     Cargado: WJets-HT400to600
     Cargado: WJets-HT100to200
     Cargado: WJets-HT200to400
     Cargado: WJets-HT70to100
    
     plots 



    
![png](output_64_1.png)
    



![png](output_64_2.png)
    



![png](output_64_3.png)
    



![png](output_64_4.png)
    

# Signal Region

## **Table 13 — Final Event Selection (SL and AH)**

### **What does it contain?**

The **official final cuts** defining the analysis **Signal Regions (SR)**.
Each SR is classified according to:

* Channel (semi-leptonic or hadronic)
* Number of **b-jets**
* Presence of **forward jets**
* Values of **MET**, **MT**, **HT**, etc.

It defines the regions where **dark matter signals** are searched for. These regions are fed directly into the **final statistical fit**.

We categorize these regions to distinguish the signal from specific backgrounds (e.g., high -tag bins suppress +jets while enhancing -like topologies). If Dark Matter exists, we expect to observe a **statistical excess of events** in the high- tails compared to the Standard Model prediction.


In [ ]:
dpoa_io.show_table("Table_13.png", "Table 13 - signal region definitions")


![png](output_67_0.png)
    


Now we move to the **signal region**. We use the events that were already filtered by the baseline selection and then apply the additional cuts described in **Table 13**. To do this, we first need to load the preprocessed datasets stored as **parquet** files, so the following function was created.

This function loads only the datasets needed for the analysis channel (muon or electron)



### Issues

The normalization is applied again at the loading stage because the stored **.parquet files do not consistently contain a final, analysis-ready weight**. In some cases, the `final_weight` column is not saved at all, and in others it may have been produced using a different luminosity, a different fileset configuration, or an earlier definition of the analysis.

To avoid mixing inconsistent normalizations and to guarantee that all samples are treated uniformly, the normalization is recomputed at runtime using the current cross sections and the total sum of generator weights. This ensures that the final event yields are coherent and directly comparable across datasets.




In [ ]:
# `load_data` is kept as a thin alias so the signal- and control-region cells
# below read exactly as they do in the documentation.
def load_data(channel="muon"):
    return load_channel(channel)


In [ ]:
def signal_regions(df):
    """Single-lepton signal regions - note Table 13.

        pTmiss          >= 160 GeV
        MT              >= 160 GeV
        MT2W            >= 200 GeV     <-- NOT IMPLEMENTED, see below
        min dphi(j1,2)  >= 1.2
        mT_b            >= 180 GeV
        categories: nb=1 & 0 FJ | nb=1 & >=1 FJ | nb>=2

    RUNBOOK PHYS-05. Upstream applied only pTmiss > 160 and min dphi > 0.5;
    MT, mT_b and the correct dphi threshold were all missing.

    MT2W IS NOT APPLIED. It requires a kinematic minimisation over jet-lepton
    assignments (note ref. [41]) that is not implemented here, so these regions
    are LOOSER than the paper's and will retain more tt(2l) background. This is
    a known, documented gap -- see RUNBOOK [OPEN-3].
    """
    if df is None or len(df) == 0:
        return {}

    pass_common = (
        (df["met"]      >= 160) &
        (df["mT_W"]     >= 160) &   # PHYS-05: was not applied at all
        (df["min_dphi"] >= 1.2) &   # PHYS-05: was 0.5
        (df["mT_b"]     >= 180)     # PHYS-05: computed, never applied
    )

    df_sr = df[pass_common]
    if len(df_sr) == 0:
        return {}

    mask_1b_0f = (df_sr["nBTag"] == 1) & (df_sr["nForwardJets"] == 0)
    mask_1b_1f = (df_sr["nBTag"] == 1) & (df_sr["nForwardJets"] >= 1)
    mask_2b    = (df_sr["nBTag"] >= 2)

    return {
        "SR_1b_0f": df_sr[mask_1b_0f],
        "SR_1b_1f": df_sr[mask_1b_1f],
        "SR_2b":    df_sr[mask_2b],
    }


Once `plot_grouped_stack` was defined, the same base code was reused to create `plot_region_stack`. The logic of the stacked histogram is the same, but extra features were added to handle analysis regions. In particular, a region label is drawn on the plot and the display is adapted to show results specific to a selected region. This keeps the core behavior unchanged while adding more flexibility for regional visualization.

The main changes are:

* A region label parameter was added to the function signature.
* A text label was drawn on the plot using a line like:
  `ax.text(..., region_label, ...)`
* The plot title or decoration was adapted to reflect the selected region.
* Small adjustments were made to make the function work with region-specific dictionaries.

All other parts of the stacking logic (histogram creation, grouping, colors, and legends) remain unchanged.





In [ ]:
from collections import OrderedDict


# --- stacking order, RUNBOOK PLOT-01 --------------------------------------
# The paper stacks processes in a FIXED physics order (bottom to top), not by
# yield. Sorting by yield puts the largest process at the bottom, which for the
# single-lepton channel (ttbar dominant) inverts the picture completely: the
# same numbers produce a stack that looks nothing like Figure 4.
# Read off the legends of Figures 4 and 5, bottom of the stack first:
PAPER_STACK_ORDER = ["Diboson", "tX", "ttbar", "WJets", "Zvv", "Zll"]


def plot_region_stack(dfs_dict, region_name, var_name, x_label, x_range,
                      channel_label, n_bins=10, y_span=(1000.0, 150.0),
                      overflow=True, lumi_label=None):
    """Stacked MC vs data with a Data/Bkg ratio panel underneath.

    Laid out like Figures 4 and 5 of the paper: log-scale main panel, ratio
    panel below with a line at 1. The upstream version had no ratio panel,
    which is the panel that actually shows whether data and simulation agree.

    RUNBOOK PLOT-01: processes are stacked in the paper's fixed order.
    RUNBOOK PLOT-03: the last bin absorbs the overflow, as the captions of
    Figures 4 and 5 state ("The last bin contains overflow events"). Without
    it our last bin falls off while the paper's rises, which is the single
    most visible shape difference in the comparison.
    """
    bins = np.linspace(x_range[0], x_range[1], n_bins + 1)
    centers = 0.5 * (bins[1:] + bins[:-1])
    hi_edge = bins[-1]

    def _hist(values, weights=None):
        v = np.asarray(values, dtype=float)
        if overflow:
            # Push everything above the last edge into the last bin. Clipping
            # just below the edge keeps np.histogram from dropping it.
            v = np.minimum(v, hi_edge - 1e-9)
        return np.histogram(v, bins=bins, weights=weights)[0]

    grouped_counts, grouped_info = {}, {}
    for name, df in dfs_dict.items():
        if name in DATA_SL:
            continue
        counts = _hist(df[var_name], weights=df["final_weight"])
        g, label, color = get_group_info(name)
        if g not in grouped_counts:
            grouped_counts[g] = counts
            grouped_info[g] = {"label": label, "color": color, "yield": counts.sum()}
        else:
            grouped_counts[g] += counts
            grouped_info[g]["yield"] += counts.sum()

    # PLOT-01: the paper's order, restricted to the groups actually present.
    order = [g for g in PAPER_STACK_ORDER if g in grouped_counts]
    order += [g for g in grouped_counts if g not in PAPER_STACK_ORDER]

    mc_counts = [grouped_counts[g] for g in order]
    mc_colors = [grouped_info[g]["color"] for g in order]
    mc_labels = [grouped_info[g]["label"] for g in order]
    total_mc = np.sum(mc_counts, axis=0) if mc_counts else np.zeros(n_bins)

    df_data = dfs_dict.get(channel_label)
    data_counts = (_hist(df_data[var_name]) if df_data is not None
                   else np.zeros(n_bins))

    fig, (ax, rax) = plt.subplots(
        2, 1, figsize=(9, 9), sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

    if mc_counts:
        hep.histplot(mc_counts, bins=bins, stack=True, histtype="fill",
                     color=mc_colors, label=mc_labels,
                     edgecolor="black", linewidth=0.7, ax=ax)
    hep.histplot(data_counts, bins=bins, histtype="errorbar", color="black",
                 label="Data", yerr=True, marker="o", markersize=5, ax=ax)

    hep.cms.label("Preliminary", data=True, lumi=lumi_label, year=2016, ax=ax)
    ax.text(0.04, 0.90, region_name, transform=ax.transAxes,
            fontsize=17, fontweight="bold", va="top")

    # RUNBOOK PLOT-05: legend in the paper's reading order -- Data, then the
    # stack from the TOP down: Z(ll), Z(nunu), W(lnu), ttbar, t+X, VV/VH.
    # Relying on the order matplotlib hands back was giving the reverse
    # (bottom-up), so the legend disagreed with the stack it described. Built
    # explicitly from `order` instead, which cannot drift.
    handles, labels = ax.get_legend_handles_labels()
    by_label = OrderedDict(zip(labels, handles))
    wanted = ["Data"] + [grouped_info[g]["label"] for g in reversed(order)]
    leg_labels = [l for l in wanted if l in by_label]
    leg_labels += [l for l in by_label if l not in leg_labels]
    ax.legend([by_label[l] for l in leg_labels], leg_labels,
              fontsize=13, ncol=2, loc="upper right")
    ax.set_ylabel("Events", fontsize=20)
    ax.set_yscale("log")
    # RUNBOOK PLOT-04: the axis spans the paper's PROPORTIONS, not its absolute
    # range. Open Data gives us 2.4-4.9 fb^-1 against the paper's 35.9, so our
    # stacks sit roughly an order of magnitude lower; pinning the paper's
    # absolute limits would shove our histogram into the bottom of an empty
    # frame and make the comparison look worse than it is. Instead the limits
    # are fixed multiples of the stack peak, chosen to reproduce the decades
    # visible in Figures 4 and 5 (about 3.5 below the peak, 2 above it for the
    # legend). Every panel in a channel therefore has the same dynamic range.
    peak = max(data_counts.max(), total_mc.max(), 1.0)
    ax.set_ylim(peak / y_span[0], peak * y_span[1])

    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(total_mc > 0, data_counts / total_mc, np.nan)
        rerr = np.where(total_mc > 0, np.sqrt(data_counts) / total_mc, np.nan)
    rax.errorbar(centers, ratio, yerr=rerr, fmt="o", color="black", markersize=5)
    rax.axhline(1.0, color="#888888", linestyle="--", linewidth=1)
    rax.set_ylim(0, 2.4)
    rax.set_ylabel("Data / Bkg", fontsize=14)
    rax.set_xlabel(x_label, fontsize=20)
    rax.set_xlim(x_range)
    rax.grid(axis="y", alpha=0.25)

    n_d, n_m = data_counts.sum(), total_mc.sum()
    if n_m:
        rax.text(0.98, 0.86, f"total Data/Bkg = {n_d / n_m:.2f}",
                 transform=rax.transAxes, ha="right", va="top", fontsize=11,
                 color="#444444")

    plt.tight_layout()
    plt.show()


In [ ]:
# --- Composition check against note Table 11 (RUNBOOK Part 6) -------------
# Table 11 is to the single-lepton channel what Table 12 is to the all-hadronic
# one. After the full SL selection it gives, as a share of total background:
#
#   tt(2l) 61.4%, W+jets 14.8%, t+X 14.1%, ttV 4.2%, VV/VH 3.5%,
#   tt(1l) 1.7%, Z(ll) 0.2%, Z(nunu) 0.1%,  and data/MC = 1.118
#
# Two things to read off it before comparing anything:
#
#  * The dominant background is DILEPTONIC ttbar, by 37 to 1 over the
#    semileptonic one. MT > 160 GeV removes genuine 1-lepton ttbar almost
#    entirely (its transverse mass has a Jacobian edge at the W mass); what
#    survives is ttbar where a second lepton was lost. So the two ttbar samples
#    are reported separately here, even though the stack plots merge them into
#    one yellow band as the paper's legend does.
#
#  * MT2W IS NOT IMPLEMENTED (RUNBOOK [OPEN-3]). Table 11 shows it removes 84%
#    of the background on its own -- more than every other signal-region cut
#    combined -- and 9.3x of tt(2l) specifically. Our regions are therefore
#    several times too full, dominated by ttbar. That is expected and
#    quantified, not a discrepancy to chase.
TABLE11 = {"ttbar-2l2nu": 61.4, "WJets": 14.8, "tX": 14.1,
           "Diboson": 3.5, "ttbar-semileptonic": 1.7,
           "Zll": 0.2, "Zvv": 0.1}


def composition_table(dfs, data_name, label):
    """Yields per process against note Table 11. ttbar is split 1l / 2l."""
    if not dfs:
        print(f"({label}: nothing loaded)")
        return
    yields = {}
    for name, df in dfs.items():
        if name in DATA_SL:
            continue
        # ttbar is kept split here: Table 11 separates them and the split is
        # the most informative number in this channel.
        key = name if name.startswith("ttbar") else get_group_info(name)[0]
        yields[key] = yields.get(key, 0.0) + float(df["final_weight"].sum())

    total_mc = sum(yields.values())
    n_data = len(dfs[data_name]) if data_name in dfs else 0

    print(f"\n=== {label} — composicion vs note Table 11 ===")
    print(f"{'process':<22} {'events':>12} {'obs %':>8} {'Table 11 %':>12}")
    print("-" * 58)
    for g in sorted(yields, key=lambda k: -yields[k]):
        pct = 100 * yields[g] / total_mc if total_mc else 0
        ref = TABLE11.get(g)
        print(f"{g:<22} {yields[g]:>12.1f} {pct:>7.1f}% "
              f"{(f'{ref:.1f}%' if ref is not None else '-'):>12}")
    print("-" * 58)
    print(f"{'total MC':<22} {total_mc:>12.1f}")
    print(f"{'data':<22} {n_data:>12d}")
    if total_mc:
        print(f"\ndata/MC = {n_data / total_mc:.3f}   (note Table 11: 1.118)")
    print("NB: MT2W no aplicado — el fondo esta ~6x alto por eso. Ver RUNBOOK Part 6.")


# The Table 11 percentages above are for the FULL signal selection, so they must
# be compared against our full selection too -- not against the baseline, whose
# composition is a different thing entirely. signal_regions() applies everything
# Table 13 asks for except MT2W, so summing its three regions reproduces the
# note's "MTb" row with that one cut missing. That is the honest comparison.
for _ch, _dn in (("muon", "SingleMuon"), ("electron", "SingleElectron")):
    _dfs = load_channel(_ch)
    if not _dfs:
        continue
    _sr = {}
    for _name, _df in _dfs.items():
        _parts = [d for d in signal_regions(_df).values() if d is not None and len(d)]
        if _parts:
            _sr[_name] = pd.concat(_parts, ignore_index=True)
    composition_table(_sr, _dn, f"{_ch} — SR completa (sin MT2W)")


def run_analysis_for_channel(channel_mode):

    raw_dfs = load_data(channel_mode)
    if not raw_dfs:
        return

    regions_db = {"SR_1b_0f": {}, "SR_1b_1f": {}, "SR_2b": {}}
    for name, df in raw_dfs.items():
        for reg, df_reg in signal_regions(df).items():
            regions_db[reg][name] = df_reg

    d_label = "SingleMuon" if channel_mode == "muon" else "SingleElectron"

    # RUNBOOK PLOT-02: paper Figure 4 bins pTmiss from 160 to 520 GeV in nine
    # 40 GeV bins, the last one holding the overflow. We had 160-600 in ten
    # 44 GeV bins, so not a single bin edge lined up with the paper's and the
    # two histograms could not be compared bin by bin.
    ch = "mu" if channel_mode == "muon" else "e"
    kw = dict(n_bins=9,
              lumi_label=round(LUMI_BY_CHANNEL[d_label] / 1000.0, 1))

    plot_region_stack(regions_db["SR_2b"], f"1{ch}, 2 b tag, SR",
                      "met", r"$p_T^{miss}$ [GeV]", (160, 520), d_label, **kw)
    plot_region_stack(regions_db["SR_1b_1f"], f"1{ch}, SR, 1 FJ",
                      "met", r"$p_T^{miss}$ [GeV]", (160, 520), d_label, **kw)
    plot_region_stack(regions_db["SR_1b_0f"], f"1{ch}, SR, 0 FJ",
                      "met", r"$p_T^{miss}$ [GeV]", (160, 520), d_label, **kw)


In [ ]:
from collections import OrderedDict


if 'sum_weights_map' in globals():
    run_analysis_for_channel("muon")
    run_analysis_for_channel("electron")
else:
    print(" Error.")


![png](output_75_0.png)
    



![png](output_75_1.png)
    



![png](output_75_2.png)
    



![png](output_75_3.png)
    



![png](output_75_4.png)
    



![png](output_75_5.png)
    

### Control Region


## **Table 14 — Control Region Definitions (SL and AH)**

### **What does it contain?**

This table defines the **Control Regions (CRs)** required to calibrate and validate the main background processes in the analysis:

* **Dileptonic tt̄**
* **W+jets**
* **Z→νν** (in the all-hadronic channel)
* **Semileptonic tt̄** (in the all-hadronic channel)

### **What is it used for?**

These regions serve to:

* **Constrain and normalize the simulated background yields**
* **Compare Monte Carlo predictions against real data**
* **Reduce systematic uncertainties**

They are essential for **controlling the dominant backgrounds** before extracting the signal in the Signal Regions (SRs).




In [ ]:
dpoa_io.show_table("Table_14.png", "Table 14 - control region definitions")


![png](output_78_0.png)
    



In [ ]:
def filter_cr_wlnu(df):
    """W+jets control region - note Table 14, columns SL1eWR / SL1mWR.

        nlep = 1,  njet >= 2,  nb = 0,  pTmiss >= 160 GeV,  MT >= 160 GeV

    The nb = 0 requirement is what makes this orthogonal to the signal region.
    Upstream used MT >= 80 GeV; Table 14 requires 160.
    """
    if df is None or len(df) == 0:
        return {}

    mask = (
        (df["met"]   >= 160) &
        (df["mT_W"]  >= 160) &   # note Table 14 (was 80)
        (df["nBTag"] == 0) &
        (df["nJet"]  >= 2)
    )

    df_cr = df[mask]
    return {"CR_Wlnu": df_cr} if len(df_cr) else {}


Once `plot_region_stack` was defined, the same base structure was reused to create `plot_cr_stack`. The core stacked histogram logic stays the same, but additional lines were added or modified to specialize the plot for Control Regions.

The main changes are:

* A **new function name** was created: `plot_cr_stack`.
* A new input parameter was added to handle the **control region label** (for example: `region_label`).
* The plot now includes a **region tag drawn on the figure**, added through a line like:
  `ax.text(..., region_label, ...)`
* The function was adapted to work with **Control Region–filtered datasets**, instead of general or signal regions.
* The default axis ranges and binning were tuned for control-region kinematics.

All histogram stacking, coloring, grouping, and legend logic remains the same as in the previous functions.



In [ ]:
def plot_cr_stack(dfs_dict, region_label, var_name, x_label, x_range, data_label, n_bins=15):

    bins = np.linspace(x_range[0], x_range[1], n_bins + 1)
    grouped_counts = {}
    grouped_info = {}

    # A) MC
    for name, df in dfs_dict.items():
        if name in DATA_SL:
            continue

        counts, _ = np.histogram(df[var_name], bins=bins, weights=df["final_weight"])
        g_key, label, color = get_group_info(name)

        if g_key not in grouped_counts:
            grouped_counts[g_key] = counts
            grouped_info[g_key] = {"label": label, "color": color, "yield": np.sum(counts)}
        else:
            grouped_counts[g_key] += counts
            grouped_info[g_key]["yield"] += np.sum(counts)

    active_groups = sorted(grouped_info.keys(), key=lambda k: grouped_info[k]["yield"])

    mc_counts = []
    mc_colors = []
    mc_labels = []
    total_mc = np.zeros(n_bins)

    for g in active_groups:
        mc_counts.append(grouped_counts[g])
        mc_colors.append(grouped_info[g]["color"])
        mc_labels.append(grouped_info[g]["label"])
        total_mc += grouped_counts[g]

    # B) DATA
    df_data = dfs_dict.get(data_label)
    if df_data is not None:
        data_counts, _ = np.histogram(df_data[var_name], bins=bins)
    else:
        data_counts = np.zeros(n_bins)

    # === Plot ===
    fig, ax = plt.subplots(figsize=(10, 8))

    hep.histplot(mc_counts, bins=bins, stack=True, histtype="fill",
                 color=mc_colors, label=mc_labels, edgecolor="black", linewidth=1, ax=ax)

    hep.histplot(data_counts, bins=bins, histtype="errorbar", color="black",
                 label=f"{data_label} (Data)", yerr=True, marker='o', markersize=5, ax=ax)

    hep.cms.label("Preliminary", data=True, lumi=round(max(LUMI_BY_CHANNEL.values()) / 1000.0, 1), year=2016, ax=ax)

    ax.text(0.05, 0.93, region_label, transform=ax.transAxes,
            fontsize=20, fontweight='bold', va='top', ha='left')

    handles, labels = ax.get_legend_handles_labels()
    by_label = OrderedDict(zip(labels[::-1], handles[::-1]))
    ax.legend(by_label.values(), by_label.keys(), fontsize=15, ncol=2, loc='upper right')

    ax.set_xlabel(x_label, fontsize=24)
    ax.set_ylabel("Events", fontsize=24)
    ax.set_xlim(x_range)
    ax.set_yscale("log")

    max_y = max(np.max(data_counts), np.max(total_mc))
    ax.set_ylim(0.1, max_y * 500)

    plt.tight_layout()
    plt.show()




In [ ]:

def run_SL(channel_mode):

    # 1. Load datasets
    raw_dfs = load_data(channel_mode)
    if not raw_dfs:
        return

    # 2. Filtrar región CR W(lν)
    cr_dict = {}
    for name, df in raw_dfs.items():
        reg = filter_cr_wlnu(df)
        if "CR_Wlnu" in reg:
            cr_dict[name] = reg["CR_Wlnu"]

    if len(cr_dict) == 0:
        print(" No events passed CR W(lν).")
        return

    # 3. Etiquetas correctas según Tabla 14
    if channel_mode == "muon":
        region_label = "SL1mWR"
        data_label = "SingleMuon"
    else:
        region_label = "SL1eWR"
        data_label = "SingleElectron"


    # plot MET
    plot_cr_stack(cr_dict, region_label, "met", r"$p_T^{miss}$ [GeV]", (160, 600), data_label, n_bins=15)

    # plot MT
    #plot_cr_stack(cr_dict, region_label, "mT_W", r"$m_T^W$ [GeV]", (160, 500), data_label, n_bins=15)


# --- RUN ---
run_SL("muon")
run_SL("electron")

![png](output_82_0.png)
    



![png](output_82_1.png)
    
